In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

In [4]:
df = pd.read_csv("../data/application_train.csv")

print("Dataset loaded successfully!")
print("Shape:", df.shape)

Dataset loaded successfully!
Shape: (307511, 122)


In [5]:
X = df.drop("TARGET", axis=1)
y = df["TARGET"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (307511, 121)
y shape: (307511,)


In [6]:
X_temp, X_test, y_temp, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp,
    y_temp,
    test_size=0.25,
    random_state=42,
    stratify=y_temp
)

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

print("y_train:", y_train.shape)
print("y_val:", y_val.shape)
print("y_test:", y_test.shape)

X_train: (184506, 121)
X_val: (61502, 121)
X_test: (61503, 121)
y_train: (184506,)
y_val: (61502,)
y_test: (61503,)


In [7]:
numerical_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

print("Numerical features:", len(numerical_features))
print("Categorical features:", len(categorical_features))
print("\nCategorical features:")
print(categorical_features)

Numerical features: 105
Categorical features: 16

Categorical features:
['NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'NAME_TYPE_SUITE', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'OCCUPATION_TYPE', 'WEEKDAY_APPR_PROCESS_START', 'ORGANIZATION_TYPE', 'FONDKAPREMONT_MODE', 'HOUSETYPE_MODE', 'WALLSMATERIAL_MODE', 'EMERGENCYSTATE_MODE']


In [8]:
train_missing = X_train.isnull().sum()
train_missing = train_missing[train_missing > 0].sort_values(ascending=False)

print("Features with missing values:", len(train_missing))
train_missing.head(20)

Features with missing values: 67


COMMONAREA_MEDI             128797
COMMONAREA_AVG              128797
COMMONAREA_MODE             128797
NONLIVINGAPARTMENTS_MEDI    127968
NONLIVINGAPARTMENTS_MODE    127968
NONLIVINGAPARTMENTS_AVG     127968
FONDKAPREMONT_MODE          126060
LIVINGAPARTMENTS_MODE       125984
LIVINGAPARTMENTS_MEDI       125984
LIVINGAPARTMENTS_AVG        125984
FLOORSMIN_MODE              125139
FLOORSMIN_MEDI              125139
FLOORSMIN_AVG               125139
YEARS_BUILD_MODE            122572
YEARS_BUILD_MEDI            122572
YEARS_BUILD_AVG             122572
OWN_CAR_AGE                 121929
LANDAREA_AVG                109294
LANDAREA_MEDI               109294
LANDAREA_MODE               109294
dtype: int64

## Feature Type and Missing Value Analysis

The dataset contains both numerical and categorical variables. Numerical features will be handled using numerical imputation, while categorical features will be imputed and encoded separately.

The training, validation, and test datasets were split before preprocessing. This prevents information from the validation or test sets from influencing the preprocessing parameters learned from the training data.

In [9]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

In [10]:
numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

In [11]:
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        drop="first"
    ))
])

In [12]:
preprocessor = ColumnTransformer([
    ("num", numerical_pipeline, numerical_features),
    ("cat", categorical_pipeline, categorical_features)
])

## Preprocessing Pipeline

Numerical features are processed using median imputation, while categorical features are processed using most-frequent-value imputation followed by one-hot encoding.

The preprocessing steps are implemented using a scikit-learn pipeline so that the imputation and encoding parameters are learned only from the training data and then applied consistently to the validation and test sets.

In [13]:
preprocessor.fit(X_train)

print("Preprocessor fitted successfully!")

Preprocessor fitted successfully!


In [14]:
X_train_processed = preprocessor.transform(X_train)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

print("X_train_processed:", X_train_processed.shape)
print("X_val_processed:", X_val_processed.shape)
print("X_test_processed:", X_test_processed.shape)

X_train_processed: (184506, 229)
X_val_processed: (61502, 229)
X_test_processed: (61503, 229)


In [16]:
print("Training missing values:",
      np.isnan(X_train_processed).sum())

print("Validation missing values:",
      np.isnan(X_val_processed).sum())

print("Test missing values:",
      np.isnan(X_test_processed).sum())

Training missing values: 0
Validation missing values: 0
Test missing values: 0


In [17]:
negative = (y_train == 0).sum()
positive = (y_train == 1).sum()

scale_pos_weight = negative / positive

print("Negative samples:", negative)
print("Positive samples:", positive)
print("Scale Pos Weight:", scale_pos_weight)

Negative samples: 169611
Positive samples: 14895
Scale Pos Weight: 11.38710976837865


In [18]:
import joblib

joblib.dump(preprocessor, "../data/preprocessor.pkl")

joblib.dump(X_train_processed, "../data/X_train_processed.pkl")
joblib.dump(X_val_processed, "../data/X_val_processed.pkl")
joblib.dump(X_test_processed, "../data/X_test_processed.pkl")

joblib.dump(y_train, "../data/y_train.pkl")
joblib.dump(y_val, "../data/y_val.pkl")
joblib.dump(y_test, "../data/y_test.pkl")

print("Preprocessing artifacts saved successfully!")

Preprocessing artifacts saved successfully!


In [19]:
joblib.dump(X_train, "../data/X_train_raw.pkl")
joblib.dump(X_val, "../data/X_val_raw.pkl")
joblib.dump(X_test, "../data/X_test_raw.pkl")

print("Raw data splits saved successfully!")

Raw data splits saved successfully!


# Preprocessing Summary

The dataset was divided into training, validation, and test sets using stratified sampling to preserve the original target-class distribution.

The final split contains:

- 184,506 training samples
- 61,502 validation samples
- 61,503 test samples

Numerical missing values were imputed using the median calculated from the training data. Categorical missing values were imputed using the most frequent category and categorical variables were transformed using one-hot encoding.

The preprocessing pipeline was fitted exclusively on the training set and subsequently applied to the validation and test sets to prevent data leakage.

After preprocessing, no missing values remained in the training, validation, or test feature matrices.

The training set contains 169,611 negative-class samples and 14,895 positive-class samples, corresponding to a class ratio of approximately 11.39:1. This imbalance will be addressed during model training using class weighting and appropriate evaluation metrics.

Both the raw data splits and preprocessing artifacts were saved for use in subsequent stages of the project.